# ES 0DTE stale quotes: results

Answers research questions 1-3 from the README. All logic lives in `src/stale_quotes/metrics.py`; this notebook only loads and displays its outputs.

**Before running:** cache the data and build the per-date tables once:

```bash
python scripts/download.py
python scripts/run_study.py        # writes data/processed/{stale,latency}/ and results/
```

Definitions (details in the `metrics` and `staleness` docstrings):

| Measure | Definition |
|---|---|
| Reaction latency | `t0` to the option's first top-of-book price change after the futures move (size-only updates ignored). The directional version needs the mid to move the way delta says. |
| Censoring | Each (event, option) pair is followed until the next futures move, the 10 ms horizon, a data gap, or expiry. Quantiles are Kaplan-Meier, so censored pairs are not dropped. Blank quantiles mean censoring stopped the curve before that level. |
| Opportunity rate | Share of futures events with at least one option stale after costs (half an option tick + $1.00 fee). |
| Stale-window duration | `t0` until the stale side is re-priced past the threshold or pulled (#6). |
| Edge | Points after costs; `capacity_usd` = edge in $ x displayed size. |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from stale_quotes import config, metrics

pd.set_option("display.float_format", "{:,.3f}".format)

stale, latency = metrics.load_processed(config.PROCESSED_DIR)
tables = metrics.build_tables(stale, latency)
print(f"{stale['trading_date'].nunique()} days, {stale.groupby(['trading_date', 'event_id']).ngroups:,} futures events, "
      f"{stale['instrument_id'].nunique()} options, {len(stale):,} (event, option) pairs")

## Data coverage

Share of pairs per status. Pairs that cannot be judged (`no_quote`, `invalid_quote`, black76 failures such as `below_intrinsic` or `low_vega`) are excluded from the rates below, so check that they are not concentrated in one moneyness bucket or time of day.

In [ ]:
display(tables["status_overall"])
display(tables["status_moneyness"])
display(tables["status_session"])

## RQ1: How quickly do option quotes react?

`zero_latency_share` is the share of reactions stamped at exactly `t0`. Futures and options run on different matching engines (issue #1), so a large share, or reactions that look faster than physically possible, means microsecond latencies should be read at a coarser horizon.

In [ ]:
display(tables["latency_overall"])
display(tables["latency_moneyness"])
display(tables["latency_session"])
display(tables["latency_move"])

In [ ]:
display(tables["directional_latency_moneyness"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics.plot_latency(latency, "moneyness", ax=axes[0])
metrics.plot_latency(latency, "session", ax=axes[1])
fig.tight_layout()

## RQ2: How often do futures moves leave a stale quote?

`event_rate` counts each futures move once, however many options it leaves stale. The hypothesis predicts low rates for sub-tick and single-tick moves and much higher rates for multi-tick moves.

In [ ]:
display(tables["opportunity_move"])
display(tables["opportunity_moneyness"])
display(tables["opportunity_session"])
metrics.plot_rate(tables["opportunity_move"], title="Futures moves leaving at least one stale option");

## RQ3: How long do stale quotes survive?

In [ ]:
display(tables["duration_overall"])
display(tables["duration_move"])
display(tables["duration_moneyness"])
display(tables["duration_session"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
metrics.plot_survival(stale, "move_bucket", ax=axes[0])
metrics.plot_survival(stale, "moneyness", ax=axes[1])
fig.tight_layout()

## Is the edge worth it after costs?

In [ ]:
display(tables["edge_overall"])
display(tables["edge_move"])
display(tables["edge_moneyness"])
metrics.plot_edge(stale);

## Answers

To be filled in from the outputs above once `run_study.py` has run on the full Oct 5-9 sample:

1. **Reaction latency:**
2. **Opportunity frequency:**
3. **Window duration:**
4. **Variation by move size, moneyness and time of day:**